# Step 3: Cart-Aware (Behavior-Aware) SASRec

**DS5500 Data Science Capstone — Behavior-Aware Sequential Recommendation for Next Purchase Prediction**

This notebook builds **directly on** `Step1_SequenceConstruction.ipynb` and
`Step2_CandidateSampling_PopularityBaseline.ipynb`.

Scope: **cart-aware SASRec only**.

Shared evaluation protocol (Step 2):
- 100 negatives sampled from **train-seen items only** + 1 positive target
- same users / targets / `item_to_idx` / cutoffs
- metrics via `evaluation_utils.py`

Cart-aware modeling:
1. Cleaned intentful-cart histories (purchase-priority `MAX_SEQ_LEN`)
2. Behavior-aware SASRec with optional log-popularity item bias
3. Optional intentful-cart boost + popularity residual at scoring (val-tuned)


## 1. Setup and shared artifacts


In [ ]:
import pickle
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from evaluation_utils import evaluate_model_dataset

ARTIFACT_DIR = Path("artifacts")
SEQUENCES_PATH = ARTIFACT_DIR / "sequences_and_splits.pkl"
CANDIDATES_PATH = ARTIFACT_DIR / "shared_evaluation_candidates.pkl"
POP_SUMMARY_PATH = ARTIFACT_DIR / "popularity_val_summary.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ---- Load Step 1 shared sequences / vocabulary / cutoffs ----
with open(SEQUENCES_PATH, "rb") as f:
    data = pickle.load(f)

train_sequences = data["train_sequences"]
val_targets = data["val_targets"]
test_targets = data["test_targets"]
val_eval_users = data["val_eval_users"]
test_eval_users = data["test_eval_users"]

item_to_idx = data["item_to_idx"]
idx_to_item = data["idx_to_item"]
all_items = data["all_items"]
n_items = data["n_items"]

config = data["config"]
MAX_SEQ_LEN = config["MAX_SEQ_LEN"]
VAL_DAYS = config["VAL_DAYS"]
TEST_DAYS = config["TEST_DAYS"]
VAL_CUTOFF = config["val_cutoff"]
TEST_CUTOFF = config["test_cutoff"]

# ---- Load Step 2 shared candidate sets (do not regenerate) ----
with open(CANDIDATES_PATH, "rb") as f:
    cand_data = pickle.load(f)

val_candidate_sets = cand_data["val_candidate_sets"]
test_candidate_sets = cand_data["test_candidate_sets"]
N_NEGATIVES = cand_data["n_negatives"]
EVAL_K = cand_data["eval_k"]
RANDOM_SEED = cand_data.get("random_seed", 42)

assert set(val_candidate_sets) == set(val_eval_users)
assert set(test_candidate_sets) == set(test_eval_users)

print("Shared data loaded from Step 1 / Step 2")
print(f"Training users: {len(train_sequences):,}")
print(f"Validation users: {len(val_eval_users):,}")
print(f"Test users: {len(test_eval_users):,}")
print(f"Vocabulary size: {n_items:,}")
print(f"MAX_SEQ_LEN={MAX_SEQ_LEN} | VAL_DAYS={VAL_DAYS} | TEST_DAYS={TEST_DAYS}")
print(f"VAL_CUTOFF={VAL_CUTOFF} | TEST_CUTOFF={TEST_CUTOFF}")
CANDIDATE_POOL = cand_data.get("candidate_pool", "unknown")
print(f"Candidates/user={N_NEGATIVES + 1} | EVAL_K={EVAL_K}")
print(f"Candidate pool: {CANDIDATE_POOL}")
assert CANDIDATE_POOL == "train_seen_items_only", (
    "Expected train-seen-only candidates from Step 2. "
    "Re-run Step2_CandidateSampling_PopularityBaseline.ipynb to regenerate artifacts."
)


## 2. Data overview (same checks as Step 2)

Confirm we are evaluating the same population and vocabulary the popularity baseline used.


In [ ]:
train_items = {
    item
    for sequence in train_sequences.values()
    for item in sequence
}
all_items_set = set(all_items)

print("Training Item Coverage")
print("-" * 40)
print(f"Total vocabulary items: {len(all_items_set):,}")
print(f"Items seen in training: {len(train_items):,}")
print(f"Items unseen in training: {len(all_items_set - train_items):,}")
print(f"Coverage: {len(train_items) / len(all_items_set) * 100:.2f}%")

coverage_rows = []
for split_name, eval_users, targets in [
    ("Validation", val_eval_users, val_targets),
    ("Test", test_eval_users, test_targets),
]:
    seen = sum(targets[user] in train_items for user in eval_users)
    unseen = len(eval_users) - seen
    coverage_rows.append({
        "Split": split_name,
        "Users": len(eval_users),
        "Seen targets": seen,
        "Unseen targets": unseen,
        "Seen target %": round(seen / len(eval_users) * 100, 2),
    })

display(pd.DataFrame(coverage_rows))

assert len(all_items_set) == n_items
assert all(item in item_to_idx for item in all_items_set)
print("Vocabulary consistency checks passed")


## 3. Popularity baseline reference (from Step 2)

Recompute popularity on the **saved train-seen-negative** candidate sets.
This should match `artifacts/popularity_val_summary.csv` (~HitRate@10 **0.3641** after the train-seen sampling change).


In [ ]:
item_popularity = Counter(
    item
    for sequence in train_sequences.values()
    for item in sequence
)

def score_candidates_popularity(candidates, item_popularity):
    return [item_popularity.get(item, 0) for item in candidates]

val_pop_scores = {
    user: score_candidates_popularity(candidates, item_popularity)
    for user, candidates in val_candidate_sets.items()
}

pop_summary, pop_results = evaluate_model_dataset(
    candidate_sets=val_candidate_sets,
    model_scores=val_pop_scores,
    targets=val_targets,
    k=EVAL_K,
)

print("Popularity Baseline — Validation Results (reference)")
print("-" * 60)
display(pd.DataFrame([pop_summary]))

if POP_SUMMARY_PATH.exists():
    saved_pop = pd.read_csv(POP_SUMMARY_PATH).iloc[0]
    for metric in ["HitRate@10", "NDCG@10", "MRR"]:
        ok = np.isclose(saved_pop[metric], pop_summary[metric])
        print(f"Match Step2 {metric}: {ok} (saved={saved_pop[metric]:.6f}, here={pop_summary[metric]:.6f})")
else:
    print("No saved popularity_val_summary.csv found; continuing with recomputed reference.")


## 4. Cart-aware model config

Shared protocol knobs stay fixed (`MAX_SEQ_LEN`, cutoffs, vocabulary, candidates).
Only cart-aware SASRec training hyperparameters are set here.


In [ ]:
# Cart-aware SASRec hyperparameters
HIDDEN_SIZE = 64
N_BLOCKS = 2
N_HEADS = 1
DROPOUT = 0.2
BATCH_SIZE = 128
LR = 1e-3
EPOCHS = 50
PATIENCE = 5
SEED = RANDOM_SEED

# Resolve the full-catalog negative-sampling popularity floor without
# changing Step 2 candidates:
USE_POPULARITY_BIAS = True          # learnable item bias, init from log1p(pop)
CART_BOOST = 1.0                    # added after per-user score standardization
# Optional residual mix with standardized log-popularity at inference
POP_RESIDUAL_WEIGHT = 0.25

# Cart parquet locations
DATA_CANDIDATES = [
    Path("data/subsampled"),
    Path("/content/subsampled"),
    Path("Desktop/DS5500/data/subsampled"),
    Path.home() / "Desktop" / "DS5500" / "data" / "subsampled",
]

OUTPUT_DIR = ARTIFACT_DIR / "step3_sasrec"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
print("Output dir:", OUTPUT_DIR.resolve())
print(
    f"USE_POPULARITY_BIAS={USE_POPULARITY_BIAS} | "
    f"CART_BOOST={CART_BOOST} | POP_RESIDUAL_WEIGHT={POP_RESIDUAL_WEIGHT}"
)


## 5. Build cleaned cart-aware histories

Load `product_buy` + `add_to_cart` + `remove_from_cart` with `timestamp < VAL_CUTOFF`
(same training window as Step 1). Map SKUs through the **shared** `item_to_idx`.

**Intentful cart filter**
- Keep every purchase
- Keep an `add_to_cart` only if the same SKU is **not** `remove_from_cart`'d before the next purchase
- Drop removes and noisy/cancelled adds

**Purchase-priority length cap (`MAX_SEQ_LEN=50`)**
- Never let carts displace purchases when truncating
- If purchases alone exceed 50, keep the most recent purchases
- Otherwise keep all purchases and fill remaining slots with the most recent intentful carts

Also store each user's **intentful cart SKUs** (original product IDs) for scoring-time boosts.


In [ ]:
BEHAVIOR_PAD = 0
BEHAVIOR_BUY = 1
BEHAVIOR_ADD = 2
BEHAVIOR_REMOVE = 3

BEHAVIOR_FILE_MAP = {
    "product_buy": ("product_buy_subsampled.parquet", BEHAVIOR_BUY),
    "add_to_cart": ("add_to_cart_subsampled.parquet", BEHAVIOR_ADD),
    "remove_from_cart": ("remove_from_cart_subsampled.parquet", BEHAVIOR_REMOVE),
}


def find_data_dir():
    return next((p for p in DATA_CANDIDATES if p.exists()), None)


def filter_intentful_events(item_ids, behavior_ids):
    """Keep purchases + adds not removed before the next buy. Drop removes/noisy carts."""
    n = len(item_ids)
    keep = [False] * n

    for i, (item, beh) in enumerate(zip(item_ids, behavior_ids)):
        if beh == BEHAVIOR_BUY:
            keep[i] = True
            continue
        if beh != BEHAVIOR_ADD:
            continue

        removed_before_next_buy = False
        for j in range(i + 1, n):
            if behavior_ids[j] == BEHAVIOR_BUY:
                break
            if behavior_ids[j] == BEHAVIOR_REMOVE and item_ids[j] == item:
                removed_before_next_buy = True
                break
        if not removed_before_next_buy:
            keep[i] = True

    out_items = [item for item, k in zip(item_ids, keep) if k]
    out_behs = [beh for beh, k in zip(behavior_ids, keep) if k]
    return out_items, out_behs


def purchase_priority_truncate(item_ids, behavior_ids, max_len):
    """Truncate to max_len without letting carts wipe purchases."""
    if len(item_ids) <= max_len:
        return item_ids, behavior_ids

    buy_idx = [i for i, b in enumerate(behavior_ids) if b == BEHAVIOR_BUY]
    cart_idx = [i for i, b in enumerate(behavior_ids) if b == BEHAVIOR_ADD]

    if len(buy_idx) >= max_len:
        selected = buy_idx[-max_len:]
    else:
        n_cart_slots = max_len - len(buy_idx)
        selected = sorted(buy_idx + cart_idx[-n_cart_slots:])

    return (
        [item_ids[i] for i in selected],
        [behavior_ids[i] for i in selected],
    )


def load_behavior_sequences(data_dir, item_to_idx, idx_to_item, val_cutoff, max_len):
    """Cleaned multi-behavior sequences before val_cutoff, shared item IDs."""
    frames = []
    for behavior, (fname, beh_id) in BEHAVIOR_FILE_MAP.items():
        path = Path(data_dir) / fname
        if not path.exists():
            raise FileNotFoundError(f"Missing {path}")
        df = pd.read_parquet(path).drop_duplicates().copy()
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
        df = df.dropna(subset=["timestamp", "client_id", "sku"])
        df = df[df["timestamp"] < val_cutoff]
        df = df[df["sku"].isin(item_to_idx)]
        df["item_id"] = df["sku"].map(item_to_idx).astype(np.int64)
        df["behavior_id"] = beh_id
        frames.append(df[["client_id", "item_id", "behavior_id", "timestamp"]])

    events = pd.concat(frames, ignore_index=True)
    events = events.sort_values(["client_id", "timestamp"]).reset_index(drop=True)

    item_seqs, beh_seqs = {}, {}
    intentful_cart_skus = {}
    stats = {
        "users": 0,
        "raw_events": 0,
        "raw_adds": 0,
        "raw_removes": 0,
        "raw_buys": 0,
        "kept_adds": 0,
        "kept_buys": 0,
        "truncated_users": 0,
    }
    purchase_users = set(train_sequences)

    for user, group in events.groupby("client_id", sort=False):
        if user not in purchase_users:
            continue
        items = group["item_id"].astype(int).tolist()
        behs = group["behavior_id"].astype(int).tolist()
        stats["users"] += 1
        stats["raw_events"] += len(items)
        stats["raw_buys"] += sum(b == BEHAVIOR_BUY for b in behs)
        stats["raw_adds"] += sum(b == BEHAVIOR_ADD for b in behs)
        stats["raw_removes"] += sum(b == BEHAVIOR_REMOVE for b in behs)

        items, behs = filter_intentful_events(items, behs)
        cart_skus = {
            int(idx_to_item[i])
            for i, b in zip(items, behs)
            if b == BEHAVIOR_ADD and i in idx_to_item
        }
        before_cap = len(items)
        items, behs = purchase_priority_truncate(items, behs, max_len)
        if before_cap > max_len:
            stats["truncated_users"] += 1

        if items:
            item_seqs[user] = items
            beh_seqs[user] = behs
            intentful_cart_skus[user] = cart_skus
            stats["kept_buys"] += sum(b == BEHAVIOR_BUY for b in behs)
            stats["kept_adds"] += sum(b == BEHAVIOR_ADD for b in behs)

    return item_seqs, beh_seqs, intentful_cart_skus, stats


data_dir = find_data_dir()
if data_dir is None:
    raise FileNotFoundError(
        "Cart-aware SASRec needs subsampled parquet files. "
        f"Looked in: {[str(p) for p in DATA_CANDIDATES]}"
    )

print("Loading cart+buy events from:", data_dir.resolve())
behavior_item_seqs, behavior_beh_seqs, intentful_cart_skus, clean_stats = load_behavior_sequences(
    data_dir, item_to_idx, idx_to_item, VAL_CUTOFF, MAX_SEQ_LEN
)

print(f"Cart-aware sequences: {len(behavior_item_seqs):,}")
print("Cleaning stats")
print("-" * 40)
print(f"Users processed: {clean_stats['users']:,}")
print(
    f"Raw events — buy={clean_stats['raw_buys']:,}, "
    f"add={clean_stats['raw_adds']:,}, remove={clean_stats['raw_removes']:,}"
)
print(
    f"Kept after intentful filter + purchase-priority cap — "
    f"buy={clean_stats['kept_buys']:,}, add={clean_stats['kept_adds']:,}"
)
print(f"Users truncated by MAX_SEQ_LEN: {clean_stats['truncated_users']:,}")
users_with_cart = sum(1 for s in intentful_cart_skus.values() if s)
print(f"Users with >=1 intentful cart SKU: {users_with_cart:,}")
if clean_stats["raw_adds"]:
    print(
        f"Intentful add keep rate: "
        f"{clean_stats['kept_adds'] / clean_stats['raw_adds'] * 100:.1f}%"
    )

assert all(BEHAVIOR_REMOVE not in behs for behs in behavior_beh_seqs.values())


## 6. Cart-aware SASRec model

SASRec backbone + behavior-type embedding + optional **log-popularity item bias**:

\(x = E_{item} + E_{behavior} + E_{pos}\)

\(score(i) = h^\top e_i + b_i\), with \(b_i\) initialized from \(\log(1+pop_i)\).

Negatives are train-seen (Step 2), so popularity is still strong on warm items but no longer helped by ultra-rare full-vocab negatives.


In [ ]:
class PointWiseFeedForward(nn.Module):
    def __init__(self, hidden_size, dropout):
        super().__init__()
        self.conv1 = nn.Conv1d(hidden_size, hidden_size, kernel_size=1)
        self.dropout1 = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(hidden_size, hidden_size, kernel_size=1)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        out = x.transpose(1, 2)
        out = self.dropout2(self.conv2(self.relu(self.dropout1(self.conv1(out)))))
        return out.transpose(1, 2)


class SASRecBlock(nn.Module):
    def __init__(self, hidden_size, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size, num_heads=n_heads, dropout=dropout, batch_first=True
        )
        self.ffn = PointWiseFeedForward(hidden_size, dropout)
        self.norm1 = nn.LayerNorm(hidden_size, eps=1e-8)
        self.norm2 = nn.LayerNorm(hidden_size, eps=1e-8)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask, key_padding_mask=None):
        h = self.norm1(x)
        attn_out, _ = self.attn(
            h, h, h, attn_mask=attn_mask, key_padding_mask=key_padding_mask, need_weights=False
        )
        x = x + self.dropout(attn_out)
        h = self.norm2(x)
        return x + self.ffn(h)


class BehaviorAwareSASRec(nn.Module):
    def __init__(
        self,
        n_items,
        n_behaviors=3,
        hidden_size=64,
        max_len=50,
        n_blocks=2,
        n_heads=1,
        dropout=0.2,
        use_popularity_bias=True,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.use_popularity_bias = use_popularity_bias
        self.item_emb = nn.Embedding(n_items + 1, hidden_size, padding_idx=0)
        self.behavior_emb = nn.Embedding(n_behaviors + 1, hidden_size, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len + 1, hidden_size, padding_idx=0)
        self.item_bias = nn.Embedding(n_items + 1, 1, padding_idx=0)
        self.emb_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [SASRecBlock(hidden_size, n_heads, dropout) for _ in range(n_blocks)]
        )
        self.final_norm = nn.LayerNorm(hidden_size, eps=1e-8)
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, (nn.Linear, nn.Embedding, nn.Conv1d)):
                nn.init.xavier_uniform_(module.weight)
            if isinstance(module, nn.Embedding) and module.padding_idx is not None:
                with torch.no_grad():
                    module.weight[module.padding_idx].fill_(0)
            if isinstance(module, nn.LayerNorm):
                nn.init.zeros_(module.bias)
                nn.init.ones_(module.weight)
        # default bias to 0; may be overwritten by init_popularity_bias()
        with torch.no_grad():
            self.item_bias.weight.zero_()

    def init_popularity_bias(self, item_popularity, item_to_idx):
        """Initialize b_i = log1p(pop) using Step1/Step2 original product IDs."""
        with torch.no_grad():
            self.item_bias.weight.zero_()
            for sku, pop in item_popularity.items():
                idx = item_to_idx.get(sku)
                if idx is None:
                    continue
                self.item_bias.weight[idx, 0] = float(np.log1p(pop))
        print(
            f"Initialized item bias from log1p(popularity) "
            f"for {len(item_popularity):,} train-seen items"
        )

    def encode(self, item_ids, behavior_ids):
        bsz, seq_len = item_ids.shape
        positions = torch.arange(1, seq_len + 1, device=item_ids.device).unsqueeze(0).expand(bsz, -1)
        positions = positions * (item_ids != 0).long()
        x = self.item_emb(item_ids) * (self.hidden_size ** 0.5)
        x = x + self.behavior_emb(behavior_ids) + self.pos_emb(positions)
        x = self.emb_dropout(x)
        pad_mask = item_ids == 0
        attn_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=item_ids.device, dtype=torch.bool), diagonal=1
        )
        for block in self.blocks:
            x = block(x, attn_mask=attn_mask, key_padding_mask=pad_mask)
        return self.final_norm(x)

    def _item_score(self, feats, item_ids):
        """feats: [B,H] or [B,L,H]; item_ids matching last dims."""
        emb = self.item_emb(item_ids)
        if feats.dim() == 2:
            scores = (feats.unsqueeze(1) * emb).sum(-1)
            if self.use_popularity_bias:
                scores = scores + self.item_bias(item_ids).squeeze(-1)
            return scores
        scores = (feats * emb).sum(-1)
        if self.use_popularity_bias:
            scores = scores + self.item_bias(item_ids).squeeze(-1)
        return scores

    def forward_logits(self, item_ids, behavior_ids, pos_ids, neg_ids):
        feats = self.encode(item_ids, behavior_ids)
        pos_logits = self._item_score(feats, pos_ids)
        neg_logits = self._item_score(feats, neg_ids)
        return pos_logits, neg_logits

    def predict_candidates(self, item_ids, behavior_ids, candidate_ids):
        feats = self.encode(item_ids, behavior_ids)[:, -1, :]
        return self._item_score(feats, candidate_ids)


def random_neq(n_items, banned, rng):
    t = int(rng.integers(1, n_items + 1))
    while t in banned:
        t = int(rng.integers(1, n_items + 1))
    return t


class CartAwareTrainDataset(Dataset):
    """Left-padded seq/pos/neg sampler; loss only on next-purchase steps."""

    def __init__(self, item_seqs, behavior_seqs, n_items, max_len=50, seed=42):
        self.item_seqs = item_seqs
        self.behavior_seqs = behavior_seqs
        self.n_items = n_items
        self.max_len = max_len
        self.rng = np.random.default_rng(seed)
        self.users = []
        for user, seq in item_seqs.items():
            if len(seq) < 2:
                continue
            beh = behavior_seqs[user]
            if any(beh[i + 1] == BEHAVIOR_BUY for i in range(len(beh) - 1)):
                self.users.append(user)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, index):
        user = self.users[index]
        ts = self.item_seqs[user]
        beh = self.behavior_seqs[user]

        seq = np.zeros(self.max_len, dtype=np.int64)
        pos = np.zeros(self.max_len, dtype=np.int64)
        neg = np.zeros(self.max_len, dtype=np.int64)
        beh_pad = np.zeros(self.max_len, dtype=np.int64)
        pos_beh = np.zeros(self.max_len, dtype=np.int64)

        nxt = ts[-1]
        nxt_beh = beh[-1]
        idx = self.max_len - 1
        banned = set(ts)
        for i in reversed(range(len(ts) - 1)):
            seq[idx] = ts[i]
            beh_pad[idx] = beh[i]
            pos[idx] = nxt
            pos_beh[idx] = nxt_beh
            neg[idx] = random_neq(self.n_items, banned, self.rng)
            nxt = ts[i]
            nxt_beh = beh[i]
            idx -= 1
            if idx == -1:
                break

        non_buy = pos_beh != BEHAVIOR_BUY
        pos[non_buy] = 0
        neg[non_buy] = 0

        return {
            "item_ids": torch.tensor(seq, dtype=torch.long),
            "behavior_ids": torch.tensor(beh_pad, dtype=torch.long),
            "pos_ids": torch.tensor(pos, dtype=torch.long),
            "neg_ids": torch.tensor(neg, dtype=torch.long),
        }


def collate(batch):
    return {k: torch.stack([b[k] for b in batch], 0) for k in batch[0]}


def sasrec_loss(pos_logits, neg_logits, pos_ids):
    mask = pos_ids != 0
    if mask.sum() == 0:
        return torch.zeros((), device=pos_logits.device)
    return F.binary_cross_entropy_with_logits(
        pos_logits[mask], torch.ones_like(pos_logits[mask])
    ) + F.binary_cross_entropy_with_logits(
        neg_logits[mask], torch.zeros_like(neg_logits[mask])
    )


def left_pad(ids, max_len):
    ids = ids[-max_len:]
    return [0] * (max_len - len(ids)) + list(ids)


print("Cart-aware SASRec ready")


## 7. Score Step 2 candidates (popularity-calibrated + cart boost)

Candidates stay **original product IDs** in Step 2 order.

Final score per candidate:

1. Cart-aware SASRec score (includes learned/log-pop item bias)
2. Per-user standardize those scores
3. Add `POP_RESIDUAL_WEIGHT * standardized log1p(popularity)`
4. Add `CART_BOOST` if the candidate is in the user's intentful cart set

Then call the same `evaluate_model_dataset` helper as Step 2.


In [ ]:
def standardize(x):
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / (x.std() + 1e-8)


def blend_scores(raw_scores, candidates, user, pop_residual_weight, cart_boost):
    """Combine model scores with popularity residual + intentful cart boost."""
    s = standardize(raw_scores)
    pop = np.log1p([item_popularity.get(sku, 0) for sku in candidates]).astype(float)
    if pop.std() > 0:
        pop = (pop - pop.mean()) / (pop.std() + 1e-8)
    carts = intentful_cart_skus.get(user, set())
    boost = np.array([cart_boost if sku in carts else 0.0 for sku in candidates], dtype=float)
    return (s + pop_residual_weight * pop + boost).tolist()


@torch.no_grad()
def score_cart_aware_on_candidates(
    model,
    item_seqs,
    behavior_seqs,
    candidate_sets,
    batch_size=256,
    pop_residual_weight=POP_RESIDUAL_WEIGHT,
    cart_boost=CART_BOOST,
):
    model.eval()
    device = next(model.parameters()).device
    users = [u for u in candidate_sets if u in item_seqs and item_seqs[u]]
    model_scores = {}

    for start in tqdm(range(0, len(users), batch_size), desc="Scoring candidates", leave=False):
        batch_users = users[start:start + batch_size]
        item_rows, beh_rows, cand_rows, cand_raw, keep = [], [], [], [], []
        for user in batch_users:
            candidates = candidate_sets[user]
            try:
                cand_idx = [item_to_idx[sku] for sku in candidates]
            except KeyError:
                continue
            item_rows.append(left_pad(item_seqs[user], MAX_SEQ_LEN))
            beh_rows.append(left_pad(behavior_seqs[user], MAX_SEQ_LEN))
            cand_rows.append(cand_idx)
            cand_raw.append(candidates)
            keep.append(user)
        if not keep:
            continue

        item_ids = torch.tensor(item_rows, dtype=torch.long, device=device)
        behavior_ids = torch.tensor(beh_rows, dtype=torch.long, device=device)
        cand_ids = torch.tensor(cand_rows, dtype=torch.long, device=device)
        scores = model.predict_candidates(item_ids, behavior_ids, cand_ids)
        for i, user in enumerate(keep):
            raw = scores[i].detach().cpu().tolist()
            model_scores[user] = blend_scores(
                raw, cand_raw[i], user, pop_residual_weight, cart_boost
            )
    return model_scores


def evaluate_cart_aware(
    model,
    item_seqs,
    behavior_seqs,
    split="val",
    model_name="behavior_aware",
    pop_residual_weight=POP_RESIDUAL_WEIGHT,
    cart_boost=CART_BOOST,
):
    if split == "val":
        candidate_sets, targets = val_candidate_sets, val_targets
    else:
        candidate_sets, targets = test_candidate_sets, test_targets

    model_scores = score_cart_aware_on_candidates(
        model,
        item_seqs,
        behavior_seqs,
        candidate_sets,
        pop_residual_weight=pop_residual_weight,
        cart_boost=cart_boost,
    )
    candidate_sets = {u: candidate_sets[u] for u in model_scores}
    targets = {u: targets[u] for u in model_scores}

    summary, user_results = evaluate_model_dataset(
        candidate_sets=candidate_sets,
        model_scores=model_scores,
        targets=targets,
        k=EVAL_K,
    )

    user_results["target_seen_in_train"] = user_results["target_item"].isin(train_items)
    coverage_results = (
        user_results.groupby("target_seen_in_train")
        .agg(
            n_users=("user_id", "count"),
            hit_rate_10=("HitRate@10", "mean"),
            ndcg_10=("NDCG@10", "mean"),
            mrr=("RR", "mean"),
        )
        .reset_index()
    )
    coverage_results["target_group"] = coverage_results["target_seen_in_train"].map({
        True: "Seen in training",
        False: "Unseen in training",
    })

    print(f"{model_name} — {split} results")
    print("-" * 45)
    display(pd.DataFrame([summary]))
    display(coverage_results[["target_group", "n_users", "hit_rate_10", "ndcg_10", "mrr"]])
    return summary, user_results, coverage_results


print("Scoring helpers ready")


## 8. Train cart-aware SASRec


In [ ]:
device = torch.device(DEVICE)
model = BehaviorAwareSASRec(
    n_items=n_items,
    n_behaviors=3,
    hidden_size=HIDDEN_SIZE,
    max_len=MAX_SEQ_LEN,
    n_blocks=N_BLOCKS,
    n_heads=N_HEADS,
    dropout=DROPOUT,
    use_popularity_bias=USE_POPULARITY_BIAS,
).to(device)

if USE_POPULARITY_BIAS:
    model.init_popularity_bias(item_popularity, item_to_idx)

dataset = CartAwareTrainDataset(
    item_seqs=behavior_item_seqs,
    behavior_seqs=behavior_beh_seqs,
    n_items=n_items,
    max_len=MAX_SEQ_LEN,
    seed=SEED,
)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.98))

print(f"[behavior_aware] train users: {len(dataset):,}")

best_ndcg = -1.0
best_path = OUTPUT_DIR / "behavior_aware_best.pt"
patience_left = PATIENCE
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    dataset.rng = np.random.default_rng(SEED + epoch)
    losses = []
    for batch in tqdm(loader, desc=f"behavior_aware epoch {epoch}", leave=False):
        item_ids = batch["item_ids"].to(device)
        behavior_ids = batch["behavior_ids"].to(device)
        pos_ids = batch["pos_ids"].to(device)
        neg_ids = batch["neg_ids"].to(device)
        optimizer.zero_grad(set_to_none=True)
        pos_logits, neg_logits = model.forward_logits(
            item_ids, behavior_ids, pos_ids, neg_ids
        )
        loss = sasrec_loss(pos_logits, neg_logits, pos_ids)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.item()))

    summary, _, _ = evaluate_cart_aware(
        model,
        behavior_item_seqs,
        behavior_beh_seqs,
        split="val",
        model_name=f"behavior_aware [epoch {epoch}]",
    )
    history.append({"epoch": epoch, "loss": float(np.mean(losses)), "val": summary})

    if summary["NDCG@10"] > best_ndcg:
        best_ndcg = summary["NDCG@10"]
        patience_left = PATIENCE
        torch.save(
            {
                "mode": "behavior_aware",
                "model_state": model.state_dict(),
                "n_items": n_items,
                "val_summary": summary,
                "epoch": epoch,
                "use_popularity_bias": USE_POPULARITY_BIAS,
                "cart_boost": CART_BOOST,
                "pop_residual_weight": POP_RESIDUAL_WEIGHT,
            },
            best_path,
        )
        print(f"[behavior_aware] saved best checkpoint -> {best_path}")
    else:
        patience_left -= 1
        if patience_left <= 0:
            print(f"[behavior_aware] early stopping at epoch {epoch}")
            break

ckpt = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state"])
print("Loaded best checkpoint from epoch", ckpt.get("epoch"))


## 9. Light validation sweep of mix weights

Keep the trained model fixed; sweep `POP_RESIDUAL_WEIGHT` and `CART_BOOST` on the
shared val candidates. This is cheap and often recovers the last gap to popularity
while preserving cart personalization.


In [ ]:
sweep_rows = []
best = None

for pop_w in [0.0, 0.25, 0.5, 1.0]:
    for cart_w in [0.0, 0.5, 1.0, 1.5]:
        summary, _, _ = evaluate_cart_aware(
            model,
            behavior_item_seqs,
            behavior_beh_seqs,
            split="val",
            model_name=f"sweep pop={pop_w} cart={cart_w}",
            pop_residual_weight=pop_w,
            cart_boost=cart_w,
        )
        row = {
            "pop_residual_weight": pop_w,
            "cart_boost": cart_w,
            **summary,
        }
        sweep_rows.append(row)
        key = (summary["NDCG@10"], summary["HitRate@10"], summary["MRR"])
        if best is None or key > best[0]:
            best = (key, row)

sweep_df = pd.DataFrame(sweep_rows).sort_values(
    ["NDCG@10", "HitRate@10", "MRR"], ascending=False
)
print("Top mix settings on shared val candidates")
display(sweep_df.head(10))

BEST_POP_RESIDUAL_WEIGHT = best[1]["pop_residual_weight"]
BEST_CART_BOOST = best[1]["cart_boost"]
print(
    f"Selected mix: POP_RESIDUAL_WEIGHT={BEST_POP_RESIDUAL_WEIGHT}, "
    f"CART_BOOST={BEST_CART_BOOST}"
)


## 10. Validation comparison vs popularity

Same users, same targets, same candidate sets, same metrics as Step 2.
Uses the selected mix weights from Section 9.


In [ ]:
summary, user_results, coverage_results = evaluate_cart_aware(
    model,
    behavior_item_seqs,
    behavior_beh_seqs,
    split="val",
    model_name="behavior_aware",
    pop_residual_weight=BEST_POP_RESIDUAL_WEIGHT,
    cart_boost=BEST_CART_BOOST,
)

comparison = pd.DataFrame([
    {"model": "popularity", **pop_summary},
    {"model": "behavior_aware", **summary},
]).set_index("model")

print("\nValidation comparison on shared Step 2 candidates")
display(comparison)

delta = summary["HitRate@10"] - pop_summary["HitRate@10"]
print(f"HitRate@10 lift vs popularity: {delta:+.4f}")

out_csv = OUTPUT_DIR / "behavior_aware_val_user_results.csv"
try:
    user_results.to_csv(out_csv, index=False)
    print("Wrote", out_csv)
except Exception as exc:
    print("CSV save skipped:", exc)

comparison.to_json(OUTPUT_DIR / "val_comparison.json", orient="index", indent=2)
sweep_df.to_json(OUTPUT_DIR / "mix_sweep_val.json", orient="records", indent=2)
print("Wrote", OUTPUT_DIR / "val_comparison.json")


## 11. Optional: test-set evaluation

Keep test reserved until model selection is done on validation, matching Step 2 guidance.


In [ ]:
RUN_TEST = False  # set True only for final numbers

if RUN_TEST:
    test_pop_scores = {
        user: score_candidates_popularity(candidates, item_popularity)
        for user, candidates in test_candidate_sets.items()
    }
    test_pop_summary, _ = evaluate_model_dataset(
        candidate_sets=test_candidate_sets,
        model_scores=test_pop_scores,
        targets=test_targets,
        k=EVAL_K,
    )
    test_summary, _, _ = evaluate_cart_aware(
        model,
        behavior_item_seqs,
        behavior_beh_seqs,
        split="test",
        model_name="behavior_aware",
        pop_residual_weight=BEST_POP_RESIDUAL_WEIGHT,
        cart_boost=BEST_CART_BOOST,
    )
    test_comparison = pd.DataFrame([
        {"model": "popularity", **test_pop_summary},
        {"model": "behavior_aware", **test_summary},
    ]).set_index("model")
    display(test_comparison)
    test_comparison.to_json(OUTPUT_DIR / "test_comparison.json", orient="index", indent=2)
else:
    print("RUN_TEST=False — validation-only comparison retained.")


## Notes

- Step 2 now samples negatives from **train-seen items only**; both popularity and cart-aware must use the regenerated `shared_evaluation_candidates.pkl`.
- Cart histories are cleaned to intentful adds and purchase-priority truncation under `MAX_SEQ_LEN`.
- Popularity bias / cart boost remain available as calibration tools, but the sampling change is the main protocol fix for fair comparison.
- Purchase-only SASRec remains out of scope here.
- Tune mix weights on val; report test only after selection.
